# Lesson 6: LSTM 声调分类教程

本教程演示如何使用 LSTM (Long Short-Term Memory) 进行声调分类。

## 学习目标
- 理解 LSTM 处理序列数据的原理
- 学习基频序列特征提取
- 掌握 PyTorch 深度学习训练流程
- 实现注意力机制和早停策略

## 1. 环境准备

In [ ]:
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from src.models.lstm_tone import LSTMToneClassifier
from src.training.lstm_trainer import LSTMTrainer

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print("环境准备完成！")

## 2. 数据加载

加载包含基频序列和声调标签的数据。

In [ ]:
# 数据路径
data_file = project_root / 'material' / 'lesson_6' / 'vowel_with_tone.csv'

if data_file.exists():
    df = pd.read_csv(data_file)
    print(f"加载数据: {len(df)} 个样本")
    print(f"\n列名: {df.columns.tolist()}")
    print(f"\n前 5 行:")
    display(df.head())
    
    # 统计声调分布
    print(f"\n声调分布:")
    print(df['tone'].value_counts().sort_index())
else:
    print(f"数据文件不存在: {data_file}")
    print("请确保数据文件在正确的位置")

## 3. 基频序列可视化

可视化不同声调的基频轮廓。

In [ ]:
if data_file.exists():
    # 假设数据中有 f0_point_0 到 f0_point_9 列
    f0_columns = [f'f0_point_{i}' for i in range(10)]
    
    if all(col in df.columns for col in f0_columns):
        fig, axes = plt.subplots(1, 5, figsize=(20, 4))
        
        for tone in range(1, 6):
            tone_data = df[df['tone'] == tone]
            
            if len(tone_data) > 0:
                # 提取基频序列
                f0_sequences = tone_data[f0_columns].values
                
                # 绘制多条轨迹
                for seq in f0_sequences[:20]:  # 只绘制前 20 条
                    axes[tone-1].plot(seq, alpha=0.3, color='blue')
                
                # 绘制平均轨迹
                mean_seq = f0_sequences.mean(axis=0)
                axes[tone-1].plot(mean_seq, color='red', linewidth=2, label='平均')
                
                axes[tone-1].set_title(f'声调 {tone}', fontsize=12)
                axes[tone-1].set_xlabel('时间步', fontsize=10)
                axes[tone-1].set_ylabel('基频 (Hz)', fontsize=10)
                axes[tone-1].legend()
                axes[tone-1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print("数据中没有基频序列列")

## 4. 数据预处理

In [ ]:
if data_file.exists():
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    
    # 提取特征和标签
    feature_columns = [col for col in df.columns if col.startswith('f0_') or col == 'duration']
    X = df[feature_columns].values
    y = df['tone'].values - 1  # 转换为 0-4
    
    # 划分数据集
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    
    # 归一化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    print(f"训练集: {len(X_train)} 个样本")
    print(f"验证集: {len(X_val)} 个样本")
    print(f"测试集: {len(X_test)} 个样本")
    print(f"特征维度: {X_train.shape[1]}")

## 5. 构建 LSTM 模型

In [ ]:
if data_file.exists():
    # 模型配置
    model_config = {
        'input_size': X_train.shape[1],
        'hidden_size': 128,
        'num_layers': 2,
        'num_classes': 5,
        'dropout': 0.3,
        'bidirectional': True,
        'use_attention': True
    }
    
    # 创建模型
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTMToneClassifier(model_config)
    model = model.to(device)
    
    print("模型架构:")
    print(model)
    
    # 统计参数量
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n总参数量: {total_params:,}")
    print(f"可训练参数: {trainable_params:,}")

## 6. 训练模型

In [ ]:
if data_file.exists():
    # 训练配置
    train_config = {
        'epochs': 50,
        'batch_size': 32,
        'learning_rate': 0.001,
        'weight_decay': 1e-5,
        'patience': 5,
        'device': device
    }
    
    # 创建训练器
    trainer = LSTMTrainer(model, train_config)
    
    # 准备数据
    # 注意：这里需要将数据重塑为 (batch, seq_len, features) 格式
    # 假设我们将所有特征作为一个时间步
    X_train_tensor = torch.FloatTensor(X_train).unsqueeze(1)  # (N, 1, features)
    y_train_tensor = torch.LongTensor(y_train)
    X_val_tensor = torch.FloatTensor(X_val).unsqueeze(1)
    y_val_tensor = torch.LongTensor(y_val)
    
    # 创建数据集
    from torch.utils.data import TensorDataset
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # 训练
    print("开始训练...")
    history = trainer.train(train_loader, val_loader)
    print("\n训练完成！")

## 7. 训练曲线可视化

In [ ]:
if data_file.exists() and 'history' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 损失曲线
    axes[0].plot(history['train_loss'], label='训练损失', marker='o')
    axes[0].plot(history['val_loss'], label='验证损失', marker='s')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('损失曲线', fontsize=14)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 准确率曲线
    axes[1].plot(history['train_acc'], label='训练准确率', marker='o')
    axes[1].plot(history['val_acc'], label='验证准确率', marker='s')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title('准确率曲线', fontsize=14)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. 模型评估

In [ ]:
if data_file.exists():
    # 测试集评估
    X_test_tensor = torch.FloatTensor(X_test).unsqueeze(1).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, y_pred = torch.max(outputs, 1)
        y_pred = y_pred.cpu().numpy()
    
    # 计算准确率
    from sklearn.metrics import accuracy_score, classification_report
    
    test_acc = accuracy_score(y_test, y_pred)
    print(f"测试集准确率: {test_acc:.4f}")
    
    print("\n分类报告:")
    tone_names = ['声调1', '声调2', '声调3', '声调4', '声调5']
    print(classification_report(y_test, y_pred, target_names=tone_names))

## 9. 混淆矩阵

In [ ]:
if data_file.exists():
    from sklearn.metrics import confusion_matrix
    
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=tone_names,
                yticklabels=tone_names)
    plt.xlabel('预测标签', fontsize=12)
    plt.ylabel('真实标签', fontsize=12)
    plt.title('混淆矩阵', fontsize=14)
    plt.tight_layout()
    plt.show()

## 10. 注意力权重可视化

如果模型使用了注意力机制，可以可视化注意力权重。

In [ ]:
if data_file.exists() and model_config.get('use_attention', False):
    # 选择一个样本
    sample_idx = 0
    sample = X_test_tensor[sample_idx:sample_idx+1]
    true_label = y_test[sample_idx]
    
    model.eval()
    with torch.no_grad():
        output = model(sample)
        pred_label = torch.argmax(output, dim=1).item()
        
        # 获取注意力权重（需要修改模型以返回注意力权重）
        # attention_weights = model.get_attention_weights(sample)
    
    print(f"真实标签: 声调{true_label+1}")
    print(f"预测标签: 声调{pred_label+1}")
    
    # 注意：实际可视化需要模型返回注意力权重

## 11. 总结

本教程演示了：
1. ✅ 加载和预处理基频序列数据
2. ✅ 构建双向 LSTM 模型（带注意力机制）
3. ✅ 训练模型并实现早停
4. ✅ 可视化训练过程
5. ✅ 评估模型性能

### 关键技术点
- **双向 LSTM**: 同时捕捉前向和后向的时序信息
- **注意力机制**: 自动学习关注重要的时间步
- **早停**: 防止过拟合
- **学习率调度**: 动态调整学习率

### 下一步
- 尝试不同的 LSTM 层数和隐藏单元数
- 实验不同的注意力机制
- 添加更多的序列特征（MFCC 序列等）
- 使用数据增强提高模型泛化能力